In [55]:
import re
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer

preprocessed_data_dir = Path("..\data\preprocessed_data")
preprocessed_data_dir.mkdir(parents=True, exist_ok=True)

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Ekaterina\AppData\Local\Temp\ipykernel_14216\1715578548.py:6: SyntaxWarning: invalid escape sequence '\d'
  preprocessed_data_dir = Path("..\data\preprocessed_data")


In [56]:
df_movies = pd.read_csv(r'..\data\movies.csv')
df_ratings = pd.read_csv(r'..\data\ratings.csv')
df_tags = pd.read_csv(r'..\data\tags.csv')
df_links = pd.read_csv(r'..\data\links.csv')

пропуски пот годам, поскольку их немного, заполним вручную, по обработке пропусков по жанрам подумать, функциии сделать воспроизводимыми, закинуть py файлик в src.

In [57]:
def extract_year(title):
    if not isinstance(title, str):
        return None

    matches = re.findall(r'\((.*?)\)', title)
        
    if not matches:
        return None

    last_skobka = matches[-1]

    year_ = re.search(r'(\d{4})', last_skobka)

    if year_:
        year = int(year_.group(1))
        return year
    
    return None


def era(year):
    if pd.isna(year):
        return 'Unknown'
    elif year < 1980:
        return 'Old'
    elif year < 1990:
        return "1980's"
    elif year < 2000:
        return "1990's"
    elif year < 2010:
        return "2000's"
    else:
        return "2020's"

def split_genres(genre_row):

    if isinstance(genre_row, list):
        return genre_row
    
    if isinstance(genre_row, str):
        return genre_row.split('|')
    
    return None


In [58]:
df_movies['year'] = df_movies['title'].apply(extract_year)
df_movies['era'] = df_movies['year'].apply(era)
df_movies['genres'] = df_movies['genres'].apply(split_genres)
df_movies.head()

,movieId,title,genres,year,era
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995.0,1990's
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995.0,1990's
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995.0,1990's
4,5,Father of the Bride Part II (1995),[Comedy],1995.0,1990's


In [59]:
df_users = df_ratings.groupby('userId').agg(
    avg_user_rating=('rating', 'mean'),
    user_rating_count=('rating', 'count')
).reset_index()
df_users.head()

,userId,avg_user_rating,user_rating_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


In [60]:
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df_movies['genres'])
genre_columns = mlb.classes_
genres_enc_df = pd.DataFrame(
    genres_encoded, 
    columns=[f'genre_{g}' for g in genre_columns],
    index=df_movies.index
)
genres_enc_df

,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9738,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9739,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
9740,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [61]:
df_movies_featd = pd.concat([df_movies, genres_enc_df], axis=1)
df_movies_featd.head()

,movieId,title,genres,year,era,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0,1990's,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1995.0,1990's,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),[Comedy],1995.0,1990's,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [62]:
df_films_ratings = df_ratings.groupby('movieId').agg(
    avg_film_rating=('rating', 'mean'),
    film_rating_count=('rating', 'count')
).reset_index()
df_films_ratings.head()

,movieId,avg_film_rating,film_rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


In [67]:
df_user_stats = df_ratings.groupby('userId').agg(
    avg_user_rating=('rating', 'mean'),
    user_rating_count=('rating', 'count')
).reset_index()
df_user_stats.head()
df_user_film_rait = df_ratings.merge(df_user_stats, 'inner', on='userId')
df_user_film_rait.head()

,userId,movieId,rating,timestamp,avg_user_rating,user_rating_count
0,1,1,4.0,964982703,4.366379,232
1,1,3,4.0,964981247,4.366379,232
2,1,6,4.0,964982224,4.366379,232
3,1,47,5.0,964983815,4.366379,232
4,1,50,5.0,964982931,4.366379,232


In [64]:
df_movies_join_rait = df_movies_featd.merge(df_films_ratings, "left", on='movieId')
df_movies_join_rait.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['movieId', 'title', 'genres', 'year', 'era', 'genre_(no genres listed)',
       'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Children',
       'genre_Comedy', 'genre_Crime', 'genre_Documentary', 'genre_Drama',
       'genre_Fantasy', 'genre_Film-Noir', 'genre_Horror', 'genre_IMAX',
       'genre_Musical', 'genre_Mystery', 'genre_Romance', 'genre_Sci-Fi',
       'genre_Thriller', 'genre_War', 'genre_Western', 'avg_film_rating',
       'film_rating_count'],
      dtype='str')>

In [70]:
df_final = df_movies_join_rait.merge(df_user_film_rait, 'inner', on='movieId')
df_final.shape

(100836, 32)